# Straight Rows Hackathon — Starter Notebook

**Bluestaq University Programs, in partnership with NDSU**

Read `../docs/Bluestaq_NDSU_Straight_Rows_Hackathon_Official.pdf` first.
This individual event is open to NDSU undergraduate U.S. citizens. The dataset
is released at kickoff; submissions close at hour 22 of the 24-hour event.

This notebook loads all feeds, explores labeled examples, runs the starter
baseline, and writes `submission.json`. Edit your identity before validating.
The baseline is deliberately incomplete; improve its attribution and evidence.

*“My tractor is drifting. Is it the Sun, an adversary, or my own hardware?”*

| Cause | Farm observation | SDA context to investigate |
|---|---|---|
| `SPACE_WEATHER` | regional degradation | elevated S4; Kp can lead farm impact |
| `JAMMING` | excess signal loss; one or several machines | weather may be quiet or active; a delayed RFI report may exist |
| `SPOOFING` | position or guidance drift | context helps distinguish concurrent causes |
| `EQUIPMENT_FAULT` | link failures or antenna dropouts | compare neighbors and regional conditions |
| `MULTIPATH` | brief recurring dips at the same place | benign reflection rather than a regional event |

Causes can overlap. Reported GNSS coordinates can be displaced; the simulated
cross-track-error channel is a separate guidance monitor, not trusted receiver truth.


In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA = Path("..") / "data"
T0 = pd.Timestamp("2026-05-04T00:00:00Z")          # scenario epoch
EVAL_START = pd.Timestamp("2026-05-06T00:00:00Z")  # hours 48-72 are unlabeled

# ---- farm-side telemetry: 72 h x 3 tractors x 1 Hz ----
tele = pd.read_csv(DATA / "farm" / "tractor_telemetry.csv.gz", parse_dates=["ts"])
print(f"telemetry: {len(tele):,} rows, columns: {list(tele.columns)}")

# ---- space-side SDA feeds ----
sgi = json.load(open(DATA / "sda" / "sgi_indices.json"))
kp = pd.DataFrame(sgi["kp"]);  kp["ts"] = pd.to_datetime(kp["ts"])
dst = pd.DataFrame(sgi["dst"]); dst["ts"] = pd.to_datetime(dst["ts"])
s4 = pd.DataFrame(json.load(open(DATA / "sda" / "scintillation_series.json"))["series"])
s4["ts"] = pd.to_datetime(s4["ts"])
advis = json.load(open(DATA / "sda" / "constellation_advisories.json"))["advisories"]
rfi = json.load(open(DATA / "sda" / "rfi_reports.json"))["reports"]

# ---- labels for the development window (hours 0-48) ----
train_eps = json.load(open(DATA / "labels" / "train_episodes.json"))["episodes"]
print(f"labeled training episodes: {len(train_eps)}   RFI reports: {len(rfi)}   advisories: {len(advis)}")


## The 72 hours at a glance
Fleet-average signal strength (farm side) against the geomagnetic Kp index
(space side). Notice the Day-2 storm — and notice that *not every* farm-side
dip lines up with a space-side spike. Those mismatches are where the other
four cause classes live.


In [ ]:
m = (tele.set_index("ts")
         .groupby("tractor_id")["mean_cn0_dbhz"]
         .resample("5min").mean().unstack(0))

fig, ax1 = plt.subplots(figsize=(13, 4.2))
for tr in m.columns:
    ax1.plot(m.index, m[tr], lw=0.9, label=tr)
ax1.set_ylabel("mean C/N0 (dB-Hz)"); ax1.set_ylim(15, 50); ax1.legend(loc="lower left")
ax2 = ax1.twinx()
ax2.step(kp["ts"], kp["kp"], where="post", color="crimson", alpha=0.55, lw=1.6)
ax2.set_ylabel("Kp (3-hourly)", color="crimson"); ax2.set_ylim(0, 9.5)
ax1.axvline(EVAL_START, color="k", ls="--", lw=1)
ax1.text(EVAL_START, 47, "  evaluation window →", fontsize=9)
for ep in train_eps:
    ax1.axvspan(pd.Timestamp(ep["start"]), pd.Timestamp(ep["end"]),
                color="orange", alpha=0.12)
ax1.set_title("Fleet signal health vs. geomagnetic activity (labeled episodes shaded)")
plt.tight_layout(); plt.savefig("overview.png", dpi=110); plt.show()


## Episode explorer
Compare an episode against neighboring machines and surrounding time.
The examples below show jamming and spoofing. Local interference can reach
multiple machines; do not assume “all three affected” proves space weather.


In [ ]:
def show_episode(ep_id):
    ep = next(e for e in train_eps if e["id"] == ep_id)
    a = pd.Timestamp(ep["start"]) - pd.Timedelta(minutes=20)
    b = pd.Timestamp(ep["end"]) + pd.Timedelta(minutes=20)
    seg = tele[(tele.ts >= a) & (tele.ts < b)]
    fig, axes = plt.subplots(3, 1, figsize=(12, 7), sharex=True)
    for tr, grp in seg.groupby("tractor_id"):
        axes[0].plot(grp.ts, grp.mean_cn0_dbhz, lw=0.8, label=tr)
        axes[1].plot(grp.ts, grp.cross_track_error_m, lw=0.8)
        axes[2].plot(grp.ts, grp.num_sats, lw=0.8)
    for ax in axes:
        ax.axvspan(pd.Timestamp(ep["start"]), pd.Timestamp(ep["end"]),
                   color="orange", alpha=0.15)
    axes[0].set_ylabel("mean C/N0"); axes[0].legend(ncols=3)
    axes[1].set_ylabel("XTE (m)"); axes[2].set_ylabel("# sats")
    axes[0].set_title(f"{ep['id']}  —  class: {ep['class']}  —  tractors: {ep['tractors']}")
    plt.tight_layout(); plt.savefig(f"{ep_id}.png", dpi=110); plt.show()

show_episode("TRN-03")   # jamming
show_episode("TRN-07")   # spoofing — XTE climbs while C/N0 stays flat


## A baseline detector to improve
The source is `../submission/baseline.py`. It uses one-minute thresholds,
a fleet comparison, and Kp. It ignores S4, RFI, advisories, and multipath,
and assigns a uniform confidence of 0.5. It is not a reference solution.
The organizer diagnostic is supporting evidence for the 100-point judging rubric.


In [ ]:
import sys
sys.path.insert(0, str((Path("..") / "submission").resolve()))
from baseline import detect

detections = detect(DATA)
print(f"baseline found {len(detections)} candidate episodes:")
for d in detections:
    print(d["start"], d["end"], d["class"], d["tractors"])


## Write and validate your submission
Replace the identity placeholders, then replace baseline detections with your
solution's output. Evidence should cite measurements, timestamps, machines,
and relevant SDA feeds; the starter evidence is intentionally incomplete.


In [ ]:
# Submissions are public: identify yourself only by GitHub username.
PARTICIPANT = {"github_handle": "your-github-username"}
REPOSITORY_URL = "https://github.com/your-github-username/your-solution-repo"
submission = {"participant": PARTICIPANT, "repository_url": REPOSITORY_URL, "detections": detections}
out = Path("..") / "submission" / "submission.json"
out.write_text(json.dumps(submission, indent=2) + "\n")
print(f"wrote {out.resolve()}")


In [ ]:
# Format validation rejects identity placeholders until you edit them.
import subprocess
result = subprocess.run([sys.executable, str(Path("..") / "submission" /
                        "validate_submission.py"), str(out)], capture_output=True, text=True)
print(result.stdout)
if result.returncode:
    print("Fix the listed problems before submitting. Format validation does not grade correctness.")


## Where to go next

- Use S4, advisories, and RFI to corroborate attribution. Reports may be delayed,
  missing, or false; a storm does not rule out local interference.
- Compare labeled development cases, including overlapping causes.
- Preserve short, repeated multipath events; minute aggregation can erase them.
- Explain uncertainty rather than treating a confidence value as a guarantee.
- Check episode boundaries and affected machines, not only cause labels.
- Reproduce your results with an exact command in your README.

Retrospective processing is allowed. Onset accuracy is not proof of real-time
performance. Code and results lock at hour 22, followed by demos and Q&A.
